In [ ]:
import digitalhub as dh
import pandas as pd
import requests
import os

In [ ]:
modes: list[str] = ["TRANSIT"]  ## Choose one or more among: "car_park", "car", "walk", "transit
zoi: str = "restrictedAv"  ## Choose between "allBologna" and "restrictedAv"
method: str = "simplified" ## Choose between "simplified" and "extended"
version: str = "20250922"

In [ ]:
if zoi not in ["allBologna", "restrictedAv"]:
    raise ValueError("zoi is not acceptable")
if method not in ["simplified", "extended"]:
    raise ValueError("method is not acceptable")
for mode in modes:
    if mode not in ["CAR_PARK", "TRANSIT", "WALK", "CAR"]:
        raise ValueError("one of modes is not acceptable")

## Prepare OTP
Define function, build, run server

In [ ]:
# Project
PROJECT = "test-otp-v2"
project = dh.get_or_create_project(PROJECT)

In [ ]:
# OTP server function
func = project.new_function(
    name=f"otp-server-{zoi}",
    kind="container",
    base_image="docker.io/opentripplanner/opentripplanner:latest")

In [ ]:
# Go to the Minio console, and create shareable links for each of the artifacts (pbf and gtfs)
GTFS = ""
if zoi == "allBologna":
    PBF = ""
elif zoi == "restrictedAv":
    PBF1 = ""
    PBF2 = ""

In [ ]:
# Run the OTP server
if zoi == "allBologna": 
    instructions = [
    "apt-get -y update; apt-get -y install curl",
    "mkdir -p /var/opentripplanner",
    f"curl -L {PBF} -o /var/opentripplanner/osm-tn-bologna.osm.pbf",
    f"curl -L {GTFS} -o /var/opentripplanner/gtfs-pts-bus-bologna.zip"
]
elif zoi == "restrictedAv":
    instructions = [
    "apt-get -y update; apt-get -y install curl",
    "mkdir -p /var/opentripplanner",
    f"curl -L {PBF1} -o /var/opentripplanner/osm-modified-tn-bologna-inside-av.osm.pbf",
    f"curl -L {PBF2} -o /var/opentripplanner/osm-tn-bologna-outside-av.osm.pbf",
    f"curl -L {GTFS} -o /var/opentripplanner/gtfs-pts-bus-bologna.zip"
]

build_run = func.run(action="build", instructions=instructions)
build_status = build_run.wait()
# After running, wait before serving!!!

In [ ]:
if build_status.status.state == "COMPLETED":
    print(build_status.status.state)
    server_run = func.run(action="serve",
                      args=["--build", "--serve"],
                      service_ports=[{"port": 8080, "target_port": 8080}],
                      service_type='ClusterIP',
                      run_as_user=60000, run_as_group=60000
                    )

## Use OTP

In [ ]:
PROJECT = "test-otp-v2"
project = dh.get_or_create_project(PROJECT)

In [ ]:
if method == "simplified":
    od_input_key = "store://test-otp-v2/dataitem/table/od-coords-simplified:s471xcn8igf703bbmfo4nn3l"
if method == "extended":
    od_input_key = "store://test-otp-v2/dataitem/table/od-coords-extended:19tqn6505owcb31amfv1qz0g"

url_port = "s-containerserve-33cb570823814262b007d59800a0f525.digitalhub-bologna:8080" ### Get it from the RUN>Service>url in DHCore
url_port = "s-containerserve-f2498d96d49944c1a39593c67ef428fa.digitalhub-bologna:8080" ### Get it from the RUN>Service>url in DHCore
modes_str =  "_".join(modes)
otp_endpoint = f"http://{url_port}/otp/gtfs/v1"
output_name = f"otp_results_{method}_{zoi}_{modes_str}_v{version}"

In [ ]:
otp_func = project.new_function("otp-processor",
                                    kind="python",
                                    python_version="PYTHON3_10",
                                    code_src="otp_processor.py",
                                    handler="main_platform")

In [ ]:
otp_run = otp_func.run("job", 
                       inputs={'df': od_input_key},
                       parameters={'main_modes': modes,
                                   'method': method,
                                   'zoi': zoi,
                                   'output_name': output_name,
                                   'otp_endpoint': otp_endpoint})